In [17]:
#%%capture
!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar"
!wget "https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar"

--2026-06-03 16:42:10--  https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.12/3.5.0/spark-sql-kafka-0-10_2.12-3.5.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 104.18.18.12, 2606:4700::6812:130c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 432335 (422K) [application/java-archive]
Saving to: ‘spark-sql-kafka-0-10_2.12-3.5.0.jar.1’

spark-sql-kafka-0-1 100%[===================>] 422.20K  --.-KB/s    in 0.03s   

2026-06-03 16:42:10 (13.7 MB/s) - ‘spark-sql-kafka-0-10_2.12-3.5.0.jar.1’ saved [432335/432335]

--2026-06-03 16:42:11--  https://repo1.maven.org/maven2/org/apache/spark/spark-streaming-kafka-0-10_2.12/3.5.0/spark-streaming-kafka-0-10_2.12-3.5.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.19.12, 104.18.18.12, 2606:4700::6812:130c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.19.12|:443... connected.
HTTP request sent,

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [5]:
%pip install --upgrade typing-extensions pydantic
%pip install mistralai==1.9.7

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [29]:
#!pip install mistralai==1.9.7
#%pip install mistralai==1.9.7


Note: you may need to restart the kernel to use updated packages.


In [4]:
#Install mistral from github

In [14]:
#!git clone https://github.com/mistralai/client-python.git
#!git clone --branch v1.9.7 --depth 1 https://github.com/mistralai/client-python.git
#cd client-python
#!pip install -e /home/jovyan/client-python/

Cloning into 'client-python'...
remote: Enumerating objects: 16063, done.
remote: Counting objects: 100% (4698/4698), done.
remote: Compressing objects: 100% (728/728), done.
remote: Total 16063 (delta 4360), reused 3971 (delta 3970), pack-reused 11365 (from 3)
Receiving objects: 100% (16063/16063), 7.68 MiB | 1.43 MiB/s, done.
Resolving deltas: 100% (11559/11559), done.
fatal: destination path 'client-python' already exists and is not an empty directory.
Obtaining file:///home/jovyan/client-python
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 14.2 MB/s eta 0:00:00
  Building editable for mistralai (pyproject.toml) ... done
  Created wheel f

In [2]:
import pandas as pd
import os
import pickle
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("OCR").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [3]:
# 1. Read Stream from Kafka
raw_stream = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "192.168.1.141:8097") \
  .option("subscribe", "input") \
  .option("startingOffsets", "latest") \
  .load()

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
processed_data = raw_stream.select(
    col("key").cast(StringType()).alias("filename"),
    col("value").alias("png_data")
)

In [6]:
#import requests
from pyspark.sql import Row
from mistralai import DocumentURLChunk, ImageURLChunk, TextChunk
import json
import base64
import os
import time
# ---------------------

# Experiment Mistral OCR : API Key
myMistalAPI = '7CAsihIaHvFFl9MW2CCJPUk8T0GBZXaP'

# Initialize Mistral client with API key
from mistralai import Mistral
api_key = 'your key'
api_key = myMistalAPI
client = Mistral(api_key=api_key)



In [8]:
def call_mistral_ocr(row: Row):

    filename = row['filename']
    png_bytes = row['png_data']
    
    # Encode image as base64 for API
    encoded = base64.b64encode(png_bytes).decode()
    base64_data_url = f"data:image/jpeg;base64,{encoded}"
    #print(base64_data_url)
    
    # Process image with OCR
    image_response = client.ocr.process(
        document=ImageURLChunk(image_url=base64_data_url),
        model="mistral-ocr-latest"
    )

    #return filename, image_response, base64_data_url
    # Convert response to JSON
    response_dict = json.loads(image_response.model_dump_json())
    
    json_string = json.dumps(response_dict["pages"][0]["markdown"], indent=4)
    
    return filename, json_string

In [9]:
!pip install jiwer

In [10]:
from jiwer import cer, wer

def evaluate_ocr_accuracy(ground_truth: str, ocr_text: str):
    """Calculate CER and WER between ground truth and OCR output."""
    character_error_rate = cer(ground_truth, ocr_text)
    word_error_rate = wer(ground_truth, ocr_text)

    #print(f"CER: {character_error_rate:.4f} ({character_error_rate*100:.2f}%)")
    #print(f"WER: {word_error_rate:.4f} ({word_error_rate*100:.2f}%)")
    return character_error_rate, word_error_rate


In [11]:
def process_batch(df, batch_id):
    """
    Function executed for every micro-batch of the Structured Stream.
    Collects the rows and processes them with the OCR function.
    """

    start_time_1 = time.time()
    
    # ⚠️ WARNING: .collect() brings ALL data to the driver program. 
    # Use for testing/low-volume only. For high-volume, consider a custom Kafka Connect Sink.
    rows = df.collect()
    
    results = []
    data_dict = {}
    count = 1
    filename = None
    
    for row in rows:
      
        print("row id = ",batch_id)

        end_time_1 = time.time()
        time_use_1 = end_time_1 - start_time_1
        filename, ocr_text = call_mistral_ocr(row)
        start_time_2 = time.time()
             
        if filename is None:
            filename = f"image_{batch_id}"

        # 1. Convert string back to a dictionary
        data_dict["filename"] = filename
        
        data_dict["message"] = ocr_text
        
        ground_truth_path = f"/home/jovyan/g{batch_id}.txt"
        print(f"ground_truth = {ground_truth_path}")

        if ground_truth_path and os.path.exists(ground_truth_path):

            with open(ground_truth_path, 'r', encoding='utf-8') as f:
                ground_truth = json.load(f) if ground_truth_path.endswith('.json') else f.read().strip()

            char_err, word_err = evaluate_ocr_accuracy(ground_truth, ocr_text) #json.dumps(ocr_text, ensure_ascii=False))
            data_dict['char_err'] = char_err
            data_dict['word_err'] = word_err
        
        #count = count+1

        end_time_2 = time.time()
        time_use_2 = end_time_2 - start_time_2
        total_time = time_use_1 + time_use_2
        data_dict['time_usage'] = total_time
        
        results.append((str(batch_id), data_dict))        
        print(f"OCR Result for {results}") # Log first 80 chars

    schema = StructType([
        StructField("filename", StringType(), False),
        StructField("ocr_text", StringType(), False) 
    ])

    if results:
        
        results_df = spark.createDataFrame(results, schema)
        #print("Create a dataframe ",results_df)

        # 3. Write results to JSON files (The Solution)
        # Use the batch_id to create a unique subdirectory for the JSON files
        batch_output_path = os.path.join("/home/jovyan/json", f"batch_{batch_id}")

        results_df.write \
            .format("json") \
            .mode("overwrite") \
            .save(batch_output_path)
        
        #print(f"Successfully wrote OCR results for Batch {batch_id} to: {batch_output_path}")
        

In [ ]:
query = processed_data.writeStream \
    .foreachBatch(process_batch) \
    .start() 

query.awaitTermination()

row id =  1
ground_truth = /home/jovyan/g1.txt
OCR Result for [('1', {'filename': 'receipt3.png', 'message': '"# ZVRA\\n\\nTAX INVOICE (ABB)\\n\\nGAGAN (THAILAND) Co., Ltd.\\n999/9 The offices at Central World\\n19th floor, Unie No ML 1907-1908\\nRama I Road, Patharwan, Bangkok 10330\\nCOMPANY TAX ID: 0-10-5-545-05834-5\\n\\nZVRA ICON STAM (BRANCH 00081)\\n299 Iconsiam Unit No. M02-03, 102 Floor"', 'char_err': 0.8607594936708861, 'word_err': 1.0, 'time_usage': 0.9989442825317383})]
row id =  2
ground_truth = /home/jovyan/g2.txt
OCR Result for [('2', {'filename': 'receipt1.png', 'message': '"PLACE FACE UP ON DASH\\nCITY OF PALO ALTO\\nNOT VALID FOR\\nONSTREET PARKING\\n\\nExpiration Date/Time\\n11:59 PM\\nAUG 19, 2024\\n\\nPurchase Date/Time: 01:34pm Aug 19, 2024\\nTotal Due: $15.00\\nTotal Paid: $15.00\\nTicket #: 00005883\\nS/N #: 520117260957\\nSetting: Permit Machines\\nMach Name: Civic Center\\nRate: Daily Parking\\nPmt Type: CC (Swipe)\\n\\n#*** -1224, Visa\\nDISPLAY FACE UP ON DA

In [15]:
for q in spark.streams.active:
    q.stop()

In [ ]:
#Mistral tutorial https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/ocr/structured_ocr.ipynb#scrollTo=po7Cukllt8za